# Comparison ExperimentBu notebook model karsilastirma kosusunu calistirir.

In [ ]:
import subprocessimport sysfrom pathlib import PathPROJECT_ROOT = Path.cwd()if not (PROJECT_ROOT / 'experiments').exists():    candidate = PROJECT_ROOT.parent    if (candidate / 'experiments').exists():        PROJECT_ROOT = candidateif str(PROJECT_ROOT) not in sys.path:    sys.path.insert(0, str(PROJECT_ROOT))print(f'Project root: {PROJECT_ROOT}')

In [ ]:
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'experiment_configs' / 'rrg_experiment.yaml'OUTPUT_PATH = PROJECT_ROOT / 'results' / 'comparison_results.json'MODELS = ['free']NUM_SAMPLES = 100print('Config:', CONFIG_PATH)print('Output:', OUTPUT_PATH)

In [ ]:
import reimport timecommand = [    sys.executable,    str(PROJECT_ROOT / 'experiments' / 'run_comparison.py'),    '--config', str(CONFIG_PATH),    '--output', str(OUTPUT_PATH),    '--models', *MODELS,    '--num-samples', str(NUM_SAMPLES),]print('Running command:')print(command)if MODELS and all(name not in ('free', 'paid') for name in MODELS):    total_steps = len(MODELS)else:    total_steps = Nonecompleted_steps = 0step_markers_seen = set()start_ts = time.time()process = subprocess.Popen(    command,    cwd=PROJECT_ROOT,    stdout=subprocess.PIPE,    stderr=subprocess.STDOUT,    text=True,    bufsize=1,)for line in process.stdout:    text_line = line.rstrip()    marker_match = re.search(r'Running\s+([\w\.-]+)\.\.\.', text_line)    if marker_match:        marker = marker_match.group(1)        if marker not in step_markers_seen:            step_markers_seen.add(marker)            completed_steps += 1    elapsed = time.time() - start_ts    elapsed_min = elapsed / 60.0    if total_steps and completed_steps > 0:        avg_per_step = elapsed / completed_steps        remaining_steps = max(0, total_steps - completed_steps)        eta_sec = avg_per_step * remaining_steps        eta_min = eta_sec / 60.0        print(f'[{completed_steps}/{total_steps}] elapsed={elapsed_min:.1f}m eta~{eta_min:.1f}m | {text_line}')    else:        known = completed_steps if completed_steps > 0 else 0        print(f'[{known}/?] elapsed={elapsed_min:.1f}m eta~unknown | {text_line}')return_code = process.wait()total_elapsed_min = (time.time() - start_ts) / 60.0print(f'Finished with code={return_code} in {total_elapsed_min:.1f}m')if return_code != 0:    raise RuntimeError(f'run_comparison failed with exit code {return_code}')

In [ ]:
import jsonimport pandas as pdif OUTPUT_PATH.exists():    payload = json.loads(OUTPUT_PATH.read_text())    rows = [        {'model_name': model_name, **scores}        for model_name, scores in payload.get('model_results', {}).items()    ]    display(pd.DataFrame(rows))    display(pd.DataFrame(payload.get('statistical_comparisons', [])))else:    print('Output file not found yet.')